# 🌿 AgroScanAI: Plant Disease Showdown (CNN vs VGG vs ResNet) 🚀

### Can we teach a machine to be a digital farmer? A deep dive into classifying plant diseases with an accuracy of over 97%!

Welcome to this comprehensive analysis where we tackle a critical real-world problem: automated plant disease detection. Early and accurate diagnosis of diseases like rust and powdery mildew can save entire harvests, ensuring food security and reducing economic losses for farmers. 🌾

In this notebook, we'll embark on a journey from raw leaf images to a highly accurate classification model. We won't just build one model; we'll stage a showdown between three powerful computer vision architectures to see which one reigns supreme!

## 🎯 What's Inside This Notebook?

Here’s a roadmap of our adventure:

#### 1. 📊 Exploratory Data Analysis (EDA)
We'll start by getting to know our dataset. How many images do we have? Is the data balanced? We'll visualize sample images and class distributions to build our intuition.

#### 2. 🔧 Preprocessing & Augmentation
Preparing our image data is crucial. We'll explore image preprocessing and augmentation techniques to create a robust training pipeline.

#### 3. 🏗️ Model #1: A Custom CNN from Scratch
We'll roll up our sleeves and build a Convolutional Neural Network from the ground up to understand the core mechanics.

#### 4. 🎨 Model #2: Fine-Tuning VGG16
Next, we'll leverage the power of Transfer Learning by fine-tuning the legendary VGG16 model. Can a pre-trained model give us a head start?

#### 5. 🏆 Model #3: Unleashing ResNet50
We'll bring out the heavyweight champion, ResNet50, known for its deep residual networks. This is where things get serious!

#### 6. 📈 Results & Showdown
Finally, we'll compare all three models based on their accuracy, loss, and classification reports. We'll visualize the results with confusion matrices and declare a winner.

#### If you find this notebook insightful or learn something new, an upvote would be greatly appreciated! 👍 It helps others discover this work. Let's get started!

In [ ]:
import kagglehub
rashikrahmanpritom_plant_disease_recognition_dataset_path = kagglehub.dataset_download('rashikrahmanpritom/plant-disease-recognition-dataset')

print('Data source import complete.')


In [ ]:
import pandas as pd
import cv2
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.applications import VGG16, ResNet50, MobileNet, InceptionV3, DenseNet201
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import *

In [ ]:
path = rashikrahmanpritom_plant_disease_recognition_dataset_path

for item in os.listdir(path):
    print(item)
    print("Is folder:", os.path.isdir(os.path.join(path, item)))

In [ ]:
def loadingData(DSpath, imagesize =(224, 224)):
    imgs, lbls = [], []
    for cls in sorted(os.listdir(DSpath)):
        clspath = os.path.join(DSpath, cls)
        if not os.path.isdir(clspath): continue
        for imgnm in os.listdir(clspath):
            img = cv2.imread(os.path.join(clspath,imgnm))
            if img is not None:
                imgs.append(cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), imagesize))
                lbls.append(cls)

    return np.array(imgs), np.array(lbls)


# Loading Data to a train path, validation path, test path

In [ ]:
base = rashikrahmanpritom_plant_disease_recognition_dataset_path
trainpth, valpth, testpth = [os.path.join(base, x) for x in ['Train/Train', 'Validation/Validation', 'Test/Test']]

# EDA

In [ ]:
# How many files found in each path
def count_images_in_folder(folder_path):
    total = 0
    for class_name in os.listdir(folder_path):
        class_folder = os.path.join(folder_path, class_name)
        total += len(os.listdir(class_folder))
    return total

train_count = count_images_in_folder(trainpth)
val_count = count_images_in_folder(valpth)
test_count = count_images_in_folder(testpth)

print(f"Train images: {train_count}")
print(f"Validation images: {val_count}")
print(f"Test images: {test_count}")

In [ ]:
# How many classes in the train path
for class_name in os.listdir(trainpth):
    class_folder = os.path.join(trainpth, class_name)
    files = os.listdir(class_folder)
    print(f"{class_name}: {len(files)} images")

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt

def show_sample_images_with_axes(data_path, classes, image_per_class=2):
    fig, axs = plt.subplots(len(classes), image_per_class, figsize=(image_per_class * 4, len(classes) * 4))

    if len(classes) == 1:
        axs = [axs]  # handle when there is only one class
    elif image_per_class == 1:
        axs = [[ax] for ax in axs]  # handle 1 column

    for i, class_name in enumerate(classes):
        class_folder = os.path.join(data_path, class_name)
        images = os.listdir(class_folder)[:image_per_class]

        for j, img_name in enumerate(images):
            img_path = os.path.join(class_folder, img_name)
            img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

            ax = axs[i][j]
            ax.imshow(img)
            ax.set_title(f"{class_name}", fontsize=10)
            ax.set_xticks([0, img.shape[1]//2, img.shape[1]-1])
            ax.set_yticks([0, img.shape[0]//2, img.shape[0]-1])
            ax.grid(True, linestyle='--', linewidth=0.5, color='gray', alpha=0.3)
            ax.tick_params(axis='both', which='major', labelsize=8)

    plt.suptitle("Sample Images from Each Class with Axes", fontsize=16)
    plt.tight_layout()
    plt.show()

show_sample_images_with_axes(trainpth, ['Healthy', 'Powdery', 'Rust'], image_per_class=3)

In [ ]:
# Visualizing classes imbalances
import os
import matplotlib.pyplot as plt
import numpy as np

class_counts = {}
for class_name in os.listdir(trainpth):
    class_folder = os.path.join(trainpth, class_name)
    if os.path.isdir(class_folder):
        class_counts[class_name] = len(os.listdir(class_folder))

classes = list(class_counts.keys())
counts = list(class_counts.values())

plt.figure(figsize=(6,4))
plt.bar(classes, counts, color=['green', 'brown', 'grey'])
plt.title('Number of Images per Class (Train)')
plt.xlabel('Class')
plt.ylabel('Image Count')
plt.grid(axis='y')

for i in range(len(classes)):
    plt.text(i, counts[i] + 3, str(counts[i]), ha='center')

plt.tight_layout()
plt.show()

# Splitting data found in each path to features(x) and target(y)

In [ ]:
x_train_origin, y_train = loadingData(trainpth)
x_val_origin, y_val = loadingData(valpth)
x_test_origin, y_test = loadingData(testpth)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)
y_test_encoded = le.transform(y_test)

In [ ]:
num_classes = len(np.unique(y_train_encoded))
print(f"{num_classes} unique classes in data.")

In [ ]:
print("Classes mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# Preprocessing

In [ ]:
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

def DataGenerator(model='CNN'):
    if model == 'VGG':
        preprocessing = vgg_preprocess
        rescale = None
    elif model == 'ResNet50':
        preprocessing = resnet_preprocess
        rescale = None
    else:
        raise ValueError("Invalid model name")

    datagen = ImageDataGenerator(
        rotation_range=20,
        validation_split=0.2,
        width_shift_range=0.2,
        height_shift_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        rescale=rescale,
        preprocessing_function=preprocessing
    )

    return datagen

# Custom CNN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import SGD

# Define your model
model_CNN = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(256, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(num_classes, activation='softmax')  
])

In [ ]:
from tensorflow.keras.optimizers import Adam
model_CNN.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=25,
    restore_best_weights=True
)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen_cnn_train = ImageDataGenerator(rescale=1.0 / 255.0)
train_generator = datagen_cnn_train.flow(x_train_origin, y_train_encoded, batch_size=64, shuffle=True)
datagen_cnn_val = ImageDataGenerator(rescale=1.0 / 255.0)
val_generator = datagen_cnn_val.flow(x_val_origin, y_val_encoded, batch_size=64, shuffle=False)

In [ ]:
history = model_CNN.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=[early_stop]
)

In [ ]:
model_CNN.save('best_cnn_model.h5')

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    plt.figure(figsize=(12, 5))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title('Accuracy Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Loss Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_training_history(history)

In [ ]:
datagen_CNN_test = ImageDataGenerator(rescale=1.0 / 255.0)
test_generator_CNN = datagen_CNN_test.flow(x_test_origin, y_test_encoded, batch_size=64, shuffle=False)

# Predict
y_pred_probs_CNN = model_CNN.predict(test_generator_CNN)
y_pred_CNN = np.argmax(y_pred_probs_CNN, axis=1)
y_true = y_test_encoded

In [ ]:
model_CNN.evaluate(test_generator_CNN)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_CNN)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - CNN Model')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

# Classification Report
class_names = ['class_0', 'class_1', 'class_2']
report = classification_report(y_true, y_pred_CNN, target_names=class_names)
print(report)

In [ ]:
num_samples = 10

indices = np.random.choice(len(x_test_origin), num_samples, replace=False)

plt.figure(figsize=(15, 10))

for i, idx in enumerate(indices):
    img = x_test_origin[idx]
    true_label = y_test_encoded[idx]
    pred_label = y_pred_CNN[idx]

    plt.subplot(2, 5, i + 1)
    plt.imshow(img.astype('uint8'))
    plt.axis('off')
    true_name = le.inverse_transform([true_label])[0]
    pred_name = le.inverse_transform([pred_label])[0]
    plt.title(f"True: {true_name} | Pred: {pred_name}", fontsize=10)
    plt.title(f"True: {true_label} | Pred: {pred_label}", fontsize=10)

plt.suptitle("Sample Predictions on Test Images", fontsize=16)
plt.tight_layout()
plt.show()

# VGG Model

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:10]:
    layer.trainable = False

for layer in base_model.layers[10:]:
    layer.trainable = True
model_VGG = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')])

In [ ]:
model_VGG.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
callbacky =[
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', save_best_only=True)
    ]

In [ ]:
datagen_vgg_train = DataGenerator(model='VGG')
train_generator_vgg = datagen_vgg_train.flow(x_train_origin, y_train_encoded, batch_size=64, shuffle=True)

datagen_vgg_val = DataGenerator(model='VGG')
val_generator_vgg = datagen_vgg_val.flow(x_val_origin, y_val_encoded, batch_size=64, shuffle=False)

In [ ]:
history = model_VGG.fit(
  train_generator_vgg,
  validation_data=val_generator_vgg,
  epochs=30,
  batch_size=64
  )

In [ ]:
import numpy as np

# Save history for later use
np.save('vgg_history.npy', history.history)

In [ ]:
model_VGG.save('vgg_model_full.h5')

In [ ]:
import matplotlib.pyplot as plt

history_dict = history.history

plt.figure(figsize=(12, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_dict['accuracy'], label='Train Accuracy')
plt.plot(history_dict['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history_dict['loss'], label='Train Loss')
plt.plot(history_dict['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
datagen_vgg_test = DataGenerator(model='VGG')
test_generator_vgg = datagen_vgg_test.flow(x_test_origin, y_test_encoded, batch_size=64, shuffle=False)

# Predict
y_pred_probs = model_VGG.predict(test_generator_vgg)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test_encoded

In [ ]:
num_samples = 10

indices = np.random.choice(len(x_test_origin), num_samples, replace=False)

plt.figure(figsize=(15, 10))

for i, idx in enumerate(indices):
    img = x_test_origin[idx]
    true_label = y_test_encoded[idx]
    pred_label = y_pred[idx]

    plt.subplot(2, 5, i + 1)
    plt.imshow(img.astype('uint8'))
    plt.axis('off')
    true_name = le.inverse_transform([true_label])[0]
    pred_name = le.inverse_transform([pred_label])[0]
    plt.title(f"True: {true_name} | Pred: {pred_name}", fontsize=10)
    plt.title(f"True: {true_label} | Pred: {pred_label}", fontsize=10)

plt.suptitle("Sample Predictions on Test Images", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
y_true = y_test_encoded

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - VGG Model')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

# Classification Report
class_names = ['class_0', 'class_1', 'class_2']  
report = classification_report(y_true, y_pred, target_names=class_names)
print(report)

# ResNet50

In [ ]:
base_model_res = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

for layer in base_model_res.layers[:100]:
    layer.trainable = False
for layer in base_model_res.layers[100:]:
    layer.trainable = True


model_RES = Sequential([
    base_model_res,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])


In [ ]:
from tensorflow.keras.optimizers import SGD

optimizer_res = SGD(learning_rate=0.01, momentum=0.9)

model_RES.compile(
    optimizer=optimizer_res,
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

In [ ]:
callbacky_res = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_resnet_model.h5', save_best_only=True)
]

In [ ]:
datagen_res_train = DataGenerator(model='ResNet50')
train_generator_res = datagen_res_train.flow(x_train_origin, y_train_encoded, batch_size=16, shuffle=True)

datagen_res_val = DataGenerator(model='ResNet50')
val_generator_res = datagen_res_val.flow(x_val_origin, y_val_encoded, batch_size=64, shuffle=False)

In [ ]:
import tensorflow as tf
tf.config.run_functions_eagerly(True)

In [ ]:
history_res = model_RES.fit(
    train_generator_res,
    validation_data=val_generator_res,
    epochs=10,
    callbacks=callbacky_res
)

In [ ]:
import numpy as np
np.save('resnet_history.npy', history_res.history)

In [ ]:
import matplotlib.pyplot as plt

history_dict = history_res.history

plt.figure(figsize=(12, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_dict['accuracy'], label='Train Accuracy')
plt.plot(history_dict['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history_dict['loss'], label='Train Loss')
plt.plot(history_dict['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
y_pred_probs_Res= model_RES.predict(x_train_origin)
y_pred_Res = np.argmax(y_pred_probs_Res, axis=1)

In [ ]:
datagen_Res_test = DataGenerator(model='ResNet50')
test_generator_Res = datagen_Res_test.flow(x_test_origin, y_test_encoded, batch_size=64, shuffle=False)

# Predict
y_pred_probs_Res = model_RES.predict(test_generator_Res)
y_pred_Res = np.argmax(y_pred_probs_Res, axis=1)
y_true = y_test_encoded

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_Res)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Res Model')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

# Classification Report
class_names = ['class_0', 'class_1', 'class_2']
report = classification_report(y_true, y_pred, target_names=class_names)
print(report)

In [ ]:
num_samples = 10

indices = np.random.choice(len(x_test_origin), num_samples, replace=False)

plt.figure(figsize=(15, 10))

for i, idx in enumerate(indices):
    img = x_test_origin[idx]
    true_label = y_test_encoded[idx]
    pred_label = y_pred_Res[idx]

    plt.subplot(2, 5, i + 1)
    plt.imshow(img.astype('uint8'))
    plt.axis('off')
    true_name = le.inverse_transform([true_label])[0]
    pred_name = le.inverse_transform([pred_label])[0]
    plt.title(f"True: {true_name} | Pred: {pred_name}", fontsize=10)
    plt.title(f"True: {true_label} | Pred: {pred_label}", fontsize=10)

plt.suptitle("Sample Predictions on Test Images", fontsize=16)
plt.tight_layout()
plt.show()

# ResNet is the best! Yayyyy ❤